###Adam Valois_this project is to build a home security system
Security Alerts:
- Motion detected when house is in "away" mode
- Door opened when house is in "away" mode
- Multiple sensors triggered simultaneously (possible break-in)
Safety Alerts:
- Temperature below 35°F (frozen pipe risk)
- Temperature above 95°F (equipment failure)
- Smoke detected (fire risk)
Comfort Notifications:
- Temperature outside comfort range (65-75°F) when home is in "home" mode
- Unusual patterns (like a door left open for >5 minutes)


## Refactoring Checklist
 
### Issues Found:
- [ ] *Monolithic functions:* run_simulation() does too much, owns the main loop, reads values, prints statuses, checks alerts, and slows time.
- [ ] *Repeat code:* temp thresholds appear in check_sensor() and Sensor.isAbnormal(). Sensor lists are defined twice in initialize_sensors() and run_simulation().
- [ ] *Silent Features:*  initialize_sensors(), check_sensor(), run_simulation() are defined but not used, printing("Hello Worls") is redundant.
- [ ] *Mixed Concerns:* sensor() mixes 3 different jobs, storing sensor data, geneating random readings, and deciding if a value is abnormal.
- [ ] *Hardcoded Values* system_state is marked as "Away" and system_mode is "AWAY" which could cause inconsistency and bugs. Temp alerts for 35 and 95 degrees as well as rand number generatos are hard coded.
 
### Priority:
1. break up monolithic code
2. remove repititive code
3. reduce mixed concerns

In [ ]:
# AFTER: Setup imports used by the rest of the notebook.
import datetime
import random


In [2]:
# Add global constants so the same values are reused everywhere.
DEFAULT_SYSTEM_MODE = "AWAY"
HOME_MODE = "HOME"
TEMP_LOW_THRESHOLD = 35
TEMP_HIGH_THRESHOLD = 95
DOOR_OPEN = "OPENED"
DOOR_CLOSED = "CLOSED"
SMOKE_DETECTED = "SMOKE"
SMOKE_CLEAR = "CLEAR"
MOTION_DETECTED = "Activity detected"
MOTION_CLEAR = "No activity"

# Build the starter dictionary-based sensor data used by the early exercise cells.
def initialize_sensors(temperature):
    sensors = [
        {"id": 1, "type": "motion", "location": "Living Room", "value": MOTION_CLEAR},
        {"id": 2, "type": "temperature", "location": "Bedroom", "value": temperature},
        {"id": 3, "type": "smoke", "location": "Kitchen", "value": SMOKE_CLEAR},
        {"id": 4, "type": "door", "location": "Front Door", "value": DOOR_CLOSED}
    ]
    system_state = {"mode": DEFAULT_SYSTEM_MODE}
    return sensors, system_state

# Small helper: one place for the temperature rule.
def is_temperature_out_of_range(temperature):
    return temperature < TEMP_LOW_THRESHOLD or temperature > TEMP_HIGH_THRESHOLD

# Small helper: motion is only an alert when the house is away.
def is_motion_alert(sensor_value, system_mode):
    return sensor_value == MOTION_DETECTED and system_mode == DEFAULT_SYSTEM_MODE

# Small helper: a door opening is only a security alert in away mode.
def is_door_alert(sensor_value, system_mode):
    return sensor_value == DOOR_OPEN and system_mode == DEFAULT_SYSTEM_MODE

# Turn one sensor object into one readable output line.
def format_sensor_reading(sensor, system_mode):
    if sensor.type == "temperature":
        status = "Critical" if sensor.isAbnormal(system_mode) else "Normal"
        return f"[READING] {sensor.location} Temperature: {sensor.value}°F ({status})"
    if sensor.type == "motion":
        return f"[READING] {sensor.location} Motion: {sensor.value}"
    if sensor.type == "smoke":
        return f"[READING] {sensor.location} Smoke: {sensor.value}"
    return f"[READING] {sensor.location}: {sensor.value}"

# Print the matching alert for a sensor if it is abnormal.
def print_alert(sensor, system_mode):
    if sensor.type == "temperature" and is_temperature_out_of_range(sensor.value):
        print(f"[ALERT!] 🚨 CRITICAL: SAFETY: Temperature out of range in {sensor.location}!")
    elif sensor.type == "motion" and is_motion_alert(sensor.value, system_mode):
        print(f"[ALERT!] 🚨 HIGH: SECURITY: Motion detected in {sensor.location} while in {system_mode} mode!")
    elif sensor.type == "door" and is_door_alert(sensor.value, system_mode):
        print(f"[ALERT!] 🚨 HIGH: SECURITY: {sensor.location} opened while in {system_mode} mode!")
    elif sensor.type == "smoke" and sensor.value == SMOKE_DETECTED:
        print(f"[ALERT!] 🚨 CRITICAL: SAFETY: Smoke detected in {sensor.location}!")

# One reporting helper keeps the print flow simple.
def print_sensor_report(sensor, system_mode):
    print(format_sensor_reading(sensor, system_mode))
    if sensor.isAbnormal(system_mode):
        print_alert(sensor, system_mode)

In [ ]:

#New sensor checking function for sensor anomolies gets passed the system state as a parameter to check for motion detection when the system is in away mode
def check_sensor(sensors, system_state):
    for sensor in sensors:
        if sensor["type"] == "temperature" and is_temperature_out_of_range(sensor["value"]):
            return "Critical: Temperature out of range!"
        if sensor["type"] == "motion" and system_state["mode"] == DEFAULT_SYSTEM_MODE:
            if sensor["value"] == MOTION_DETECTED:
                return "Alert: Motion Detected!"
    return None

In [4]:
# Represent each sensor as an object with behavior instead of a plain dictionary.
class Sensor:
    def __init__(self, sensor_id, sensor_type, location, initial_value):
        self.id = sensor_id
        self.type = sensor_type
        self.location = location
        self.value = initial_value

    def read(self):
        import random
        if self.type == "door":
            self.value = random.choice([DOOR_OPEN, DOOR_CLOSED])
        elif self.type == "smoke":
            self.value = random.choice([SMOKE_CLEAR, SMOKE_DETECTED])
        elif self.type == "temperature":
            self.value = random.randint(0, 110)
        elif self.type == "motion":
            self.value = random.choice([MOTION_CLEAR, MOTION_DETECTED])

    def isAbnormal(self, system_mode):
        if self.type == "temperature":
            return is_temperature_out_of_range(self.value)
        elif self.type == "smoke":
            return self.value == SMOKE_DETECTED
        elif self.type == "motion":
            return is_motion_alert(self.value, system_mode)
        elif self.type == "door":
            return is_door_alert(self.value, system_mode)
        return False

In [5]:
import time  # Used only for pacing the demo loop.

# Run a short simulation so the notebook stays interactive.
def run_simulation(iterations=1):
    # Create a tiny sample set of sensors for the simulation.
    sensors = [
        Sensor(1, "door", "Front Door", DOOR_CLOSED),
        Sensor(2, "smoke", "Bedroom", SMOKE_CLEAR)
    ]
    system_mode = DEFAULT_SYSTEM_MODE

    # Repeat the demo a limited number of times.
    for _ in range(iterations):
        print(f"=== HomeGuard Security System ===")
        print(f"Mode: {system_mode}")

        for sensor in sensors:
            sensor.read()  # Refresh the simulated reading.
            print_sensor_report(sensor, system_mode)

        time.sleep(2)  # Pause briefly so repeated demo output is readable.

In [7]:

# Output generation: this block demonstrates the refactored helpers in one place.
demo_timestamp = datetime.datetime.now()
demo_temperature = 110
_starter_sensors, system_state = initialize_sensors(demo_temperature)

print(f"=== HomeGuard Security System ===")
print(f"Time: {demo_timestamp}")

# Create the class-based sensor objects used by the refactored flow.
demo_sensors = [
    Sensor(1, "motion", "Living Room", MOTION_CLEAR),
    Sensor(2, "temperature", "Bedroom", demo_temperature),
    Sensor(3, "smoke", "Kitchen", SMOKE_CLEAR),
    Sensor(4, "door", "Front Door", DOOR_CLOSED)
]
system_mode = system_state["mode"]
print(f"Mode: {system_mode}\n")

# Print each sensor reading and any matching alert.
for sensor in demo_sensors:
    sensor.read()
    print_sensor_report(sensor, system_mode)

=== HomeGuard Security System ===
Time: 2026-07-20 12:52:31.666520
Mode: AWAY

[READING] Living Room Motion: Activity detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[READING] Bedroom Temperature: 37°F (Normal)
[READING] Kitchen Smoke: CLEAR
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
